# LabelInspect - CPU baseline

Run this with the GPU **off**. It generates synthetic fixed-layout labels and checks the template-residual evaluation pipeline. It does not reproduce the diffusion paper. Real photographed labels and the diffusion implementation are later milestones.

Dependencies: NumPy and Pillow. The baseline is embedded below so no project upload or API key is needed.

In [ ]:
from pathlib import Path
import sys, subprocess
import numpy
import PIL
work = Path('/kaggle/working/labelinspect') if Path('/kaggle/working').exists() else Path.cwd() / 'labelinspect'
work.mkdir(parents=True, exist_ok=True)
script = work / 'baseline.py'
script.write_text('"""CPU baseline and deterministic synthetic pipeline check, not a paper reproduction."""\nfrom __future__ import annotations\nimport argparse\nimport csv\nimport json\nimport time\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont, ImageFilter\n\n\ndef font(size=20):\n    for name in (\'DejaVuSans.ttf\', \'C:/Windows/Fonts/arial.ttf\'):\n        try:\n            return ImageFont.truetype(name, size)\n        except OSError:\n            pass\n    return ImageFont.load_default(size=size)\n\n\ndef reference_label():\n    im = Image.new(\'L\', (360, 224), 238)\n    d = ImageDraw.Draw(im)\n    d.rectangle((12, 12, 347, 211), outline=30, width=2)\n    d.text((28, 28), \'QUALITY CONTROL\', font=font(25), fill=20)\n    d.text((28, 75), \'ITEM A-104\', font=font(24), fill=20)\n    d.text((28, 112), \'BATCH 2026-01\', font=font(20), fill=25)\n    rng = np.random.default_rng(100)\n    x = 30\n    while x < 270:\n        width = int(rng.integers(1, 4))\n        d.rectangle((x, 156, x + width, 193), fill=25)\n        x += width + int(rng.integers(2, 5))\n    return im\n\n\ndef make_sample(base, kind, rng):\n    altered = base.copy()\n    d = ImageDraw.Draw(altered)\n    original = np.asarray(base)\n    if kind == \'missing_print\':\n        ys, xs = np.where((original[75:105, 28:200] < 70))\n        k = int(rng.integers(len(xs)))\n        x, y = int(xs[k] + 28), int(ys[k] + 75)\n        d.rectangle((x - 2, y - 4, x + 5, y + 6), fill=238)\n    elif kind == \'smudge\':\n        x, y = int(rng.integers(190, 290)), int(rng.integers(70, 125))\n        d.ellipse((x, y, x + 22, y + 13), fill=int(rng.integers(30, 100)))\n    elif kind == \'tear\':\n        y = int(rng.integers(55, 165))\n        d.polygon([(340,y-10),(359,y-20),(359,y+22),(325,y+7)], fill=100)\n    elif kind != \'good\':\n        raise ValueError(f\'Unknown sample type: {kind}\')\n    mask = np.asarray(altered) != original\n    signal = np.asarray(altered, dtype=np.float32)\n    # Nuisance variation is not counted as a defect. Layout remains aligned.\n    signal = signal * rng.uniform(.9, 1.05) + rng.normal(0, 1.5, signal.shape)\n    return Image.fromarray(np.clip(signal,0,255).astype(\'uint8\')), Image.fromarray(mask.astype(\'uint8\')*255)\n\n\ndef generate_dataset(root, seed=230224):\n    root = Path(root)\n    base = reference_label()\n    rows = []\n    specs = [(\'train\',\'good\',48),(\'val\',\'good\',16)] + [(\'test\',k,12) for k in [\'good\',\'missing_print\',\'smudge\',\'tear\']]\n    for group,(split,kind,count) in enumerate(specs):\n        rng = np.random.default_rng(np.random.SeedSequence([seed,group]))\n        for i in range(count):\n            image, mask = make_sample(base,kind,rng)\n            relative = Path(split)/kind/f\'{i:03d}.png\'\n            target = root/relative\n            target.parent.mkdir(parents=True,exist_ok=True)\n            image.save(target)\n            mask_path = \'\'\n            if split == \'test\':\n                relmask = Path(\'masks\')/kind/f\'{i:03d}.png\'\n                (root/relmask).parent.mkdir(parents=True,exist_ok=True)\n                mask.save(root/relmask)\n                mask_path=relmask.as_posix()\n            rows.append({\'split\':split,\'kind\':kind,\'image\':relative.as_posix(),\'mask\':mask_path})\n    (root/\'manifest.json\').write_text(json.dumps({\'synthetic\':True,\'seed\':seed,\'generator_version\':1,\'samples\':rows},indent=2))\n    return rows\n\n\ndef read_normalized(path):\n    with Image.open(path) as im:\n        x=np.asarray(im.convert(\'L\'),dtype=np.float32)/255.\n    # White-background label assumption; real photographs need registration first.\n    return np.clip(x / max(float(np.median(x)),1e-6),0,1)\n\n\ndef score_image(image,template):\n    if image.shape != template.shape:\n        raise ValueError(f\'Shape mismatch: {image.shape} != {template.shape}\')\n    squared = (image-template)**2\n    # This baseline explicitly quantizes residuals to 8 bits before median filtering.\n    # Saved scores retain all resulting levels, rather than only a binary mask.\n    quantized = Image.fromarray(np.clip(squared*255,0,255).astype(\'uint8\'))\n    return np.asarray(quantized.filter(ImageFilter.MedianFilter(3)),dtype=np.float32)/255.\n\n\ndef fit_template(paths):\n    if not paths:\n        raise ValueError(\'Normal training images are required\')\n    return np.median(np.stack([read_normalized(p) for p in paths]),axis=0)\n\n\ndef calibrate_threshold(normal_scores,target_fpr=.005):\n    if not 0 < target_fpr < 1:\n        raise ValueError(\'target_fpr must be strictly between 0 and 1\')\n    if not normal_scores:\n        raise ValueError(\'Normal validation scores are required\')\n    return float(np.quantile(np.concatenate([s.ravel() for s in normal_scores]),1-target_fpr,method=\'higher\'))\n\n\ndef mask_metrics(pred,truth):\n    pred,truth=np.asarray(pred,dtype=bool),np.asarray(truth,dtype=bool)\n    if pred.shape != truth.shape:\n        raise ValueError(\'Prediction and reference shapes must match\')\n    tp=int(np.sum(pred & truth));fp=int(np.sum(pred & ~truth));fn=int(np.sum(~pred & truth));tn=int(np.sum(~pred & ~truth))\n    return {\'tp\':tp,\'fp\':fp,\'fn\':fn,\'tn\':tn,\'dice\':2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 1.,\'iou\':tp/(tp+fp+fn) if tp+fp+fn else 1.,\'precision\':tp/(tp+fp) if tp+fp else None,\'recall\':tp/(tp+fn) if tp+fn else None}\n\n\ndef preview(rows,root,template,threshold,output):\n    selected = [next(r for r in rows if r[\'split\']==\'test\' and r[\'kind\']==k) for k in [\'good\',\'missing_print\',\'smudge\',\'tear\']]\n    sheet=Image.new(\'RGB\',(1180,1110),\'#edf2f4\');d=ImageDraw.Draw(sheet)\n    d.text((30,20),\'LabelInspect | CPU baseline pipeline check\',fill=\'#172c38\',font=font(27))\n    d.text((30,60),\'SYNTHETIC DATA ONLY - not diffusion output or real-world validation\',fill=\'#9d351c\',font=font(18))\n    for c,title in enumerate([\'Input\',\'Known synthetic mask\',\'Template baseline mask\']):\n        d.text((30+c*380,104),title,fill=\'#172c38\',font=font(19))\n    for idx,row in enumerate(selected):\n        y=150+idx*235\n        image=Image.open(root/row[\'image\']).convert(\'RGB\')\n        truth=Image.open(root/row[\'mask\']).convert(\'RGB\')\n        pred=score_image(read_normalized(root/row[\'image\']),template)>threshold\n        pred=Image.fromarray(pred.astype(\'uint8\')*255).convert(\'RGB\')\n        for c,im in enumerate([image,truth,pred]):sheet.paste(im.resize((342,213)),(30+c*380,y))\n        d.text((30,y-20),row[\'kind\'],fill=\'#172c38\',font=font(15))\n    sheet.save(output)\n\n\ndef run_demo(output,seed=230224):\n    output=Path(output);root=output/\'data\';output.mkdir(parents=True,exist_ok=True)\n    rows=generate_dataset(root,seed)\n    template=fit_template([root/r[\'image\'] for r in rows if r[\'split\']==\'train\'])\n    threshold=calibrate_threshold([score_image(read_normalized(root/r[\'image\']),template) for r in rows if r[\'split\']==\'val\'])\n    np.save(output/\'template.npy\',template)\n    results=[];elapsed=[]\n    for row in rows:\n        if row[\'split\']!=\'test\':continue\n        start=time.perf_counter()\n        scores=score_image(read_normalized(root/row[\'image\']),template)\n        pred=scores>threshold;elapsed.append((time.perf_counter()-start)*1000)\n        truth=np.asarray(Image.open(root/row[\'mask\']).convert(\'L\'))>0\n        results.append({\'image\':row[\'image\'],\'kind\':row[\'kind\'],**mask_metrics(pred,truth)})\n        dest=output/\'predictions\'/row[\'kind\'];dest.mkdir(parents=True,exist_ok=True)\n        name=Path(row[\'image\']).name\n        Image.fromarray(pred.astype(\'uint8\')*255).save(dest/name)\n        np.save(dest/(Path(name).stem+\'_scores.npy\'),scores)\n    defects=[r for r in results if r[\'kind\']!=\'good\'];normal=[r for r in results if r[\'kind\']==\'good\']\n    report={\'status\':\'synthetic_pipeline_check_only\',\'method\':\'illumination_normalized_median_template_squared_residual_3x3_median\',\'seed\':seed,\'train_count\':48,\'normal_validation_count\':16,\'test_count\':48,\'test_defective_count\':36,\'test_normal_count\':12,\'threshold_source\':\'normal_validation_only\',\'target_normal_pixel_fpr\':.005,\'threshold\':threshold,\'mean_dice_defective_images\':float(np.mean([r[\'dice\'] for r in defects])),\'mean_iou_defective_images\':float(np.mean([r[\'iou\'] for r in defects])),\'normal_pixel_fpr\':sum(r[\'fp\'] for r in normal)/sum(r[\'fp\']+r[\'tn\'] for r in normal),\'median_ms_including_image_load\':float(np.median(elapsed)),\'limitations\':[\'Synthetic, perfectly aligned fixed layout\',\'No real-world accuracy claim\',\'Not DTU-Net or Tsimplex\',\'Synthetic threshold cannot be assumed valid for real data\']}\n    (output/\'metrics.json\').write_text(json.dumps(report,indent=2))\n    with (output/\'per_image.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=list(results[0]));writer.writeheader();writer.writerows(results)\n    preview(rows,root,template,threshold,output/\'preview.png\')\n    print(json.dumps(report,indent=2));return report\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'command\',choices=[\'demo\'])\n    parser.add_argument(\'--output\',default=\'artifacts/synthetic_check\')\n    parser.add_argument(\'--seed\',type=int,default=230224)\n    args=parser.parse_args();run_demo(args.output,args.seed)\n', encoding='utf8')
print('Workspace:', work)
print('NumPy:', numpy.__version__, 'Pillow:', PIL.__version__)


In [ ]:
output = work / 'artifacts' / 'synthetic_check'
subprocess.run([sys.executable, str(script), 'demo', '--output', str(output)], check=True)


In [ ]:
from IPython.display import display, Image
display(Image(filename=str(output / 'preview.png')))


## Interpretation
The masks and metrics are for synthetic, aligned labels only. A strong score here establishes that this simple case works; it says nothing about real photographed-label generalization or DTU-Net performance. The template uses training images and the threshold uses normal validation images. The held-out reference masks are used only to evaluate the final predictions.

For real data, capture and split physical labels independently, then test registration, lighting variation, and small-defect sensitivity. Save output files before ending a hosted session.